# Navigation

#  [1. Setup](#1-setup)
#  [2. Config](#2-config)
#  [3. Load Data](#3-load-data)
#  [4. Feature Engineering](#4-feature-engineering)
#  [5. Training (XGBoost + TE + PL)](#5-training-xgboost--te--pl)
#  [6. Save Outputs](#6-save-outputs)

# 1. Setup

In [1]:
try:
    get_ipython().run_line_magic("load_ext", "cudf.pandas")
    print("cudf.pandas loaded ✅ — GPU-accelerated pandas active")
except Exception as e:
    print(f"cudf.pandas unavailable ({e.__class__.__name__}) — using standard pandas ✅")

import numpy as np
import pandas as pd
import warnings, gc, time

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

cudf.pandas loaded ✅ — GPU-accelerated pandas active


# 2. Config

In [2]:
CONFIG = {
    "N_FOLDS": 10,
    "INNER_FOLDS": 10,
    "RANDOM_SEED": 77,
    "TARGET": "Churn",
    "PSEUDO_LABELS": True,
    "TRES": 0.995,
}

XGB_PARAMS = {
    "n_estimators": 50000,
    "learning_rate": 0.005,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "gamma": 0.05,
    "random_state": CONFIG["RANDOM_SEED"],
    "early_stopping_rounds": 500,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "enable_categorical": True,
    "device": "cuda",
    "tree_method": "hist",
    "verbosity": 0,
}

print("=" * 70)
print("CONFIGURATION:")
for k, v in CONFIG.items():
    print(f"{k:<20} : {v}")
print(f"\nXGB early_stopping_rounds : {XGB_PARAMS['early_stopping_rounds']}")
print("=" * 70)

CONFIGURATION:
N_FOLDS              : 10
INNER_FOLDS          : 10
RANDOM_SEED          : 77
TARGET               : Churn
PSEUDO_LABELS        : True
TRES                 : 0.995

XGB early_stopping_rounds : 500


# 3. Load Data

In [3]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIGINAL_PATH = "/kaggle/input/datasets/thedrzee/customer-churn-in-telecom-sample-dataset-by-ibm/WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("[1/3] Loading competition train set...")
train = pd.read_csv(TRAIN_PATH)
print(f"Shape: {train.shape} | Memory: {train.memory_usage(deep=True).sum()/1024**2:.2f} MB")

print("[2/3] Loading competition test set...")
test = pd.read_csv(TEST_PATH)
print(f"Shape: {test.shape}")

print("[3/3] Loading original IBM Telco dataset...")
orig = pd.read_csv(ORIGINAL_PATH)
print(f"Shape: {orig.shape}")

train[CONFIG["TARGET"]] = train[CONFIG["TARGET"]].astype(str).str.strip().map({"No": 0, "Yes": 1})
orig[CONFIG["TARGET"]] = orig[CONFIG["TARGET"]].astype(str).str.strip().map({"No": 0, "Yes": 1})

train[CONFIG["TARGET"]] = train[CONFIG["TARGET"]].fillna(0).astype("int8")
orig[CONFIG["TARGET"]] = orig[CONFIG["TARGET"]].fillna(0).astype("int8")

if "TotalCharges" in orig.columns:
    orig["TotalCharges"] = pd.to_numeric(orig["TotalCharges"], errors="coerce")
    orig["TotalCharges"] = orig["TotalCharges"].fillna(orig["TotalCharges"].median())

if "customerID" in orig.columns:
    orig.drop(columns=["customerID"], inplace=True)

train_ids = train["id"].copy()
test_ids = test["id"].copy()

print(f"\nTrain: {len(train):,} rows | Churn rate: {train[CONFIG['TARGET']].mean()*100:.2f}%")
print(f"Orig : {len(orig):,} rows | Churn rate: {orig[CONFIG['TARGET']].mean()*100:.2f}%")
print(f"Test : {len(test):,} rows")
print("=" * 70)

[1/3] Loading competition train set...
Shape: (594194, 21) | Memory: 112.86 MB
[2/3] Loading competition test set...
Shape: (254655, 20)
[3/3] Loading original IBM Telco dataset...
Shape: (7043, 21)

Train: 594,194 rows | Churn rate: 22.52%
Orig : 7,043 rows | Churn rate: 26.54%
Test : 254,655 rows


# 4. Feature Engineering

In [4]:
CATS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]
NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]

NEW_NUMS = []
NEW_CATS = []
NUM_AS_CAT = []
NON_TE_CATS = []
TO_REMOVE = []

for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f"FREQ_{col}"] = df[col].map(freq).fillna(0).astype("float32")
    NEW_NUMS.append(f"FREQ_{col}")

for df in [train, test, orig]:
    df["charges_deviation"] = (df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]).astype("float32")
    df["monthly_to_total_ratio"] = (df["MonthlyCharges"] / (df["TotalCharges"] + 1)).astype("float32")
    df["avg_monthly_charges"] = (df["TotalCharges"] / (df["tenure"] + 1)).astype("float32")
NEW_NUMS += ["charges_deviation", "monthly_to_total_ratio", "avg_monthly_charges"]

SERVICE_COLS = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
]
for df in [train, test, orig]:
    df["service_count"] = (df[SERVICE_COLS] == "Yes").sum(axis=1).astype("float32")
    df["has_internet"] = (df["InternetService"] != "No").astype("float32")
    df["has_phone"] = (df["PhoneService"] == "Yes").astype("float32")
NEW_NUMS += ["service_count", "has_internet", "has_phone"]

for df in [train, test, orig]:
    for col in NUMS:
        v = pd.to_numeric(df[col], errors="coerce")
        df[f"LOG1P_{col}"] = np.log1p(v.clip(lower=0)).astype("float32")
        df[f"SQRT_{col}"] = np.sqrt(v.clip(lower=0)).astype("float32")
NEW_NUMS += [f"LOG1P_{c}" for c in NUMS] + [f"SQRT_{c}" for c in NUMS]

_all = pd.concat([train[NUMS], test[NUMS], orig[NUMS]], axis=0, ignore_index=True)
for col in NUMS:
    r = _all[col].rank(method="average", pct=True)
    n_tr, n_te = len(train), len(test)
    train[f"RANK_{col}"] = r.iloc[:n_tr].astype("float32").values
    test[f"RANK_{col}"] = r.iloc[n_tr:n_tr + n_te].astype("float32").values
    orig[f"RANK_{col}"] = r.iloc[n_tr + n_te:].astype("float32").values
NEW_NUMS += [f"RANK_{c}" for c in NUMS]

TENURE_BINS = [0, 1, 3, 6, 12, 24, 36, 48, 60, 72, 10_000]
for df in [train, test, orig]:
    df["tenure_bin"] = pd.cut(df["tenure"], bins=TENURE_BINS, include_lowest=True).astype(str).astype("category")
NEW_CATS.append("tenure_bin")

for df in [train, test, orig]:
    df["tenure_x_monthly"] = (df["tenure"] * df["MonthlyCharges"]).astype("float32")
    df["tenure_x_service"] = (df["tenure"] * df["service_count"]).astype("float32")
NEW_NUMS += ["tenure_x_monthly", "tenure_x_service"]

for df in [train, test, orig]:
    for col in NUMS:
        df[f"MISS_{col}"] = pd.to_numeric(df[col], errors="coerce").isna().astype("int8")
NEW_NUMS += [f"MISS_{c}" for c in NUMS]

YN_COLS = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "MultipleLines",
]
for df in [train, test, orig]:
    for c in YN_COLS:
        if c in df.columns:
            df[f"ISYES_{c}"] = (df[c] == "Yes").astype("float32")
            df[f"ISNO_{c}"] = (df[c] == "No").astype("float32")
NEW_NUMS += [f"ISYES_{c}" for c in YN_COLS if c in train.columns] + [f"ISNO_{c}" for c in YN_COLS if c in train.columns]

CROSS_PAIRS = [
    ("Contract", "InternetService"),
    ("PaymentMethod", "Contract"),
    ("InternetService", "OnlineSecurity"),
]
for a, b in CROSS_PAIRS:
    if a in train.columns and b in train.columns:
        name = f"{a}__{b}"
        for df in [train, test, orig]:
            df[name] = (df[a].astype(str) + "|" + df[b].astype(str)).astype("category")
        NEW_CATS.append(name)

for col in CATS + NUMS:
    tmp = orig.groupby(col)[CONFIG["TARGET"]].mean()
    _name = f"ORIG_proba_{col}"
    train = train.merge(tmp.rename(_name), on=col, how="left")
    test = test.merge(tmp.rename(_name), on=col, how="left")
    orig = orig.merge(tmp.rename(_name), on=col, how="left")
    for df in [train, test, orig]:
        df[_name] = df[_name].fillna(0.5).astype("float32")
    NEW_NUMS.append(_name)

for col in NUMS:
    _new = f"CAT_{col}"
    NUM_AS_CAT.append(_new)
    for df in [train, test, orig]:
        df[_new] = df[col].astype(str).astype("category")

FEATURES = NUMS + CATS + NEW_NUMS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS

# 5. Training (XGBoost + TE + PL)

In [5]:
TE_COLUMNS = NUM_AS_CAT + CATS + NEW_CATS
TO_REMOVE = NUM_AS_CAT + CATS + NEW_CATS
STATS = ["std", "min", "max"]

np.random.seed(CONFIG["RANDOM_SEED"])
skf_outer = StratifiedKFold(n_splits=CONFIG["N_FOLDS"], shuffle=True, random_state=CONFIG["RANDOM_SEED"])
skf_inner = StratifiedKFold(n_splits=CONFIG["INNER_FOLDS"], shuffle=True, random_state=CONFIG["RANDOM_SEED"])

xgb_oof = np.zeros(len(train), dtype=np.float32)
xgb_pred = np.zeros(len(test), dtype=np.float32)
xgb_fold_scores = []
xgb_importances = []
xgb_pred_folds = []
pl_results = []

t0 = time.time()
print(f"\n{'='*80}")
print("TRAINING XGBOOST")
print("=" * 80)

for i, (train_idx, val_idx) in enumerate(skf_outer.split(train, train[CONFIG["TARGET"]])):
    print(f"\nFold {i+1}/{CONFIG['N_FOLDS']}")

    X_tr = train.loc[train_idx, FEATURES + [CONFIG["TARGET"]]].reset_index(drop=True).copy()
    y_tr = train.loc[train_idx, CONFIG["TARGET"]].values
    X_val = train.loc[val_idx, FEATURES].reset_index(drop=True).copy()
    y_val = train.loc[val_idx, CONFIG["TARGET"]].values
    X_te = test[FEATURES].reset_index(drop=True).copy()

    for j, (in_tr, in_va) in enumerate(skf_inner.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr, FEATURES + [CONFIG["TARGET"]]].copy()
        X_va2 = X_tr.loc[in_va, FEATURES].copy()
        for col in TE_COLUMNS:
            tmp = X_tr2.groupby(col, observed=False)[CONFIG["TARGET"]].agg(STATS)
            tmp.columns = [f"TE1_{col}_{s}" for s in STATS]
            X_va2 = X_va2.merge(tmp, on=col, how="left")
            for c in tmp.columns:
                X_tr.loc[in_va, c] = X_va2[c].values.astype("float32")

    for col in TE_COLUMNS:
        tmp = X_tr.groupby(col, observed=False)[CONFIG["TARGET"]].agg(STATS)
        tmp.columns = [f"TE1_{col}_{s}" for s in STATS]
        tmp = tmp.astype("float32")
        X_val = X_val.merge(tmp, on=col, how="left")
        X_te = X_te.merge(tmp, on=col, how="left")
        for c in tmp.columns:
            for df in [X_tr, X_val, X_te]:
                df[c] = df[c].fillna(0)

    TE_MEAN_COLS = [f"TE_{col}" for col in TE_COLUMNS]
    te = TargetEncoder(
        cv=CONFIG["INNER_FOLDS"],
        shuffle=True,
        smooth="auto",
        target_type="binary",
        random_state=CONFIG["RANDOM_SEED"],
    )
    X_tr[TE_MEAN_COLS] = te.fit_transform(X_tr[TE_COLUMNS], y_tr)
    X_val[TE_MEAN_COLS] = te.transform(X_val[TE_COLUMNS])
    X_te[TE_MEAN_COLS] = te.transform(X_te[TE_COLUMNS])

    for df in [X_tr, X_val, X_te]:
        df[CATS + NUM_AS_CAT] = df[CATS + NUM_AS_CAT].astype(str).astype("category")
        df.drop(columns=TO_REMOVE, inplace=True)
    X_tr.drop(columns=[CONFIG["TARGET"]], inplace=True)
    COLS_XGB = X_tr.columns

    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    if CONFIG["PSEUDO_LABELS"]:
        oof_p = model.predict_proba(X_val)[:, 1]
        test_p = model.predict_proba(X_te)[:, 1]
        mask = (test_p > CONFIG["TRES"]) | (test_p < 1 - CONFIG["TRES"])
        base_auc = roc_auc_score(y_val, oof_p)

        X_tr_pl = pd.concat([X_tr, X_te[mask]], axis=0)
        y_tr_pl = np.concatenate([y_tr, (test_p[mask] > 0.5).astype(int)])

        model2 = xgb.XGBClassifier(**XGB_PARAMS)
        model2.fit(X_tr_pl, y_tr_pl, eval_set=[(X_val, y_val)], verbose=False)
        oof_p2 = model2.predict_proba(X_val)[:, 1]
        pl_auc = roc_auc_score(y_val, oof_p2)

        if pl_auc > base_auc:
            model = model2
            pl_results.append(("improved", base_auc, pl_auc))
        else:
            pl_results.append(("no_gain", base_auc, pl_auc))
        del X_tr_pl, y_tr_pl, model2

    xgb_oof[val_idx] = model.predict_proba(X_val)[:, 1].astype(np.float32)
    fold_auc = roc_auc_score(y_val, xgb_oof[val_idx])
    xgb_fold_scores.append(fold_auc)

    fold_test_p = model.predict_proba(X_te[COLS_XGB])[:, 1].astype(np.float32)
    xgb_pred += fold_test_p / CONFIG["N_FOLDS"]
    xgb_pred_folds.append(fold_test_p)
    xgb_importances.append(model.get_booster().get_score(importance_type="gain"))

    print(f"Fold {i+1} AUC: {fold_auc:.5f}")

    del X_tr, X_val, X_te, y_tr, y_val, model
    gc.collect()

xgb_mean = float(np.mean(xgb_fold_scores))
xgb_std = float(np.std(xgb_fold_scores))
xgb_auc = float(roc_auc_score(train[CONFIG["TARGET"]], xgb_oof))

print(f"\n{'='*70}")
print(f"XGBoost Fold AUC : {xgb_mean:.5f} ± {xgb_std:.5f}")
print(f"XGBoost OOF AUC  : {xgb_auc:.5f}")
print(f"Wall time        : {(time.time()-t0)/60:.1f} min")
print(f"Pseudo label     : {sum(1 for r in pl_results if r[0]=='improved')}/{CONFIG['N_FOLDS']} folds improved")
print("=" * 70)


TRAINING XGBOOST

Fold 1/10
Fold 1 AUC: 0.91892

Fold 2/10
Fold 2 AUC: 0.91835

Fold 3/10
Fold 3 AUC: 0.91713

Fold 4/10
Fold 4 AUC: 0.91906

Fold 5/10
Fold 5 AUC: 0.91601

Fold 6/10
Fold 6 AUC: 0.91776

Fold 7/10
Fold 7 AUC: 0.91929

Fold 8/10
Fold 8 AUC: 0.91889

Fold 9/10
Fold 9 AUC: 0.91937

Fold 10/10
Fold 10 AUC: 0.92014

XGBoost Fold AUC : 0.91849 ± 0.00116
XGBoost OOF AUC  : 0.91849
Wall time        : 113.3 min
Pseudo label     : 0/10 folds improved


# 6. Save Outputs

In [6]:
pd.DataFrame({"xgb_oof": xgb_oof}).to_csv("oof_predictions.csv", index=False)

fold_test_df = pd.DataFrame({f"fold_{i}": p for i, p in enumerate(xgb_pred_folds)})
fold_test_df["mean"] = xgb_pred
fold_test_df.to_csv("test_predictions_per_fold.csv", index=False)

sub = pd.DataFrame({"id": test_ids, CONFIG["TARGET"]: xgb_pred})
sub.to_csv("submission.csv", index=False)

print(f"OOF AUC          : {xgb_auc:.5f}")
print(f"Fold AUC         : {xgb_mean:.5f} ± {xgb_std:.5f}")
print(f"Pred range       : [{xgb_pred.min():.5f}, {xgb_pred.max():.5f}]")
print(f"Mean churn prob  : {xgb_pred.mean():.5f}")
print(f"Features used    : {len(COLS_XGB)}")
print("Saved: submission.csv, oof_predictions.csv, test_predictions_per_fold.csv")

OOF AUC          : 0.91849
Fold AUC         : 0.91849 ± 0.00116
Pred range       : [0.00008, 0.98839]
Mean churn prob  : 0.21885
Features used    : 159
Saved: submission.csv, oof_predictions.csv, test_predictions_per_fold.csv
